# InsightVLM — NB6: Zero-Shot Open VLM Baselines

Can a general-purpose VLM name the statistical finding with no task training?
Each model sees the chart plus all 20 findings (name + one-line definition) and
must answer with one label. Greedy decoding, same prompt for every model.

Evaluated on **all 1,192 real test charts** plus a fixed 400-chart synthetic sample.
Each model is saved as soon as it finishes, so a timeout keeps completed ones.
Output: `zs_<model>.csv`, `nb6_results.json`.

In [ ]:
!pip -q install -U transformers accelerate qwen-vl-utils

In [ ]:
import os, json, time, re, gc
from pathlib import Path
import numpy as np, pandas as pd, torch
from PIL import Image

DATA_ROOT = Path("/kaggle/input/datasets/asifxzaman/xxxyyy/dataset/dataset")
DATA = DATA_ROOT if (DATA_ROOT / "records.jsonl").exists() else \
       list(Path("/kaggle/input").glob("**/records.jsonl"))[0].parent
meta = pd.read_parquet(DATA / "tables" / "metadata.parquet")
te = meta[meta.split == "test"]
EVAL = pd.concat([te[te.provenance == "real"],
                  te[te.provenance == "synthetic"].sample(400, random_state=0)]).reset_index(drop=True)
INFO = json.loads((DATA / "class_index.json").read_text())["classes"]
CLASSES = list(INFO)
print(f"eval {len(EVAL)} charts ({(EVAL.provenance=='real').sum()} real)")
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

PROMPT = ("You are a statistician. Look at the chart and decide which ONE statistical "
          "finding it shows. Options:\n" +
          "\n".join(f"- {c}: {INFO[c]['desc']}" for c in CLASSES) +
          "\nAnswer with exactly one option name from the list (e.g. dist_normal) and nothing else.")

def parse(txt):
    t = txt.strip().lower()
    hits = [c for c in CLASSES if c in t]
    if hits: return min(hits, key=t.index)          # first-mentioned label
    for c in CLASSES:                                # tolerate spaces instead of underscores
        if c.replace("_", " ") in t: return c
    return None

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

MODELS = ["Qwen/Qwen2.5-VL-7B-Instruct", "Qwen/Qwen3-VL-8B-Instruct", "Qwen/Qwen2.5-VL-3B-Instruct"]
RESULTS = {}

def run(model_id, dtype=torch.float16):
    # 768x576 charts are already inside the default pixel budget, no resize knobs needed
    proc = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForImageTextToText.from_pretrained(
        model_id, torch_dtype=dtype, device_map="auto").eval()
    rows, t0 = [], time.time()
    for k, r in EVAL.iterrows():
        msgs = [{"role": "user", "content": [{"type": "image", "image": str(DATA / r.image)},
                                             {"type": "text", "text": PROMPT}]}]
        inp = proc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True,
                                       return_dict=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=16, do_sample=False)
        txt = proc.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0]
        rows.append({"sample_id": r.sample_id, "true": r.label, "pred": parse(txt),
                     "provenance": r.provenance, "raw": txt})
        if k == 9 and sum(x["pred"] is None for x in rows) >= 8 and dtype == torch.float16:
            # T4 has no bf16; some Qwen builds overflow in fp16 and emit garbage
            print(f"  fp16 output unparseable ({rows[-1]['raw']!r}), retrying in bf16", flush=True)
            del model; gc.collect(); torch.cuda.empty_cache()
            return run(model_id, torch.bfloat16)
        if k % 100 == 0:
            el = time.time() - t0
            print(f"  {model_id} {k}/{len(EVAL)}  {el/60:.0f}m  eta {el/(k+1)*(len(EVAL)-k-1)/60:.0f}m", flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(rows)

for mid in MODELS:
    try:
        d = run(mid)
    except Exception as e:                      # a missing/unsupported model must not sink the rest
        print(f"SKIP {mid}: {type(e).__name__}: {e}"); continue
    tag = mid.split("/")[-1]
    d.to_csv(f"zs_{tag}.csv", index=False)
    d["ok"] = d.true == d.pred
    RESULTS[tag] = {"real": float(d[d.provenance == "real"].ok.mean()),
                    "synthetic": float(d[d.provenance == "synthetic"].ok.mean()),
                    "unparsed": float(d.pred.isna().mean()), "n": len(d)}
    print(tag, RESULTS[tag], flush=True)
    Path("nb6_results.json").write_text(json.dumps(RESULTS, indent=2))